# Pipeline safety explorer — incidents by state

PHMSA pipeline incidents aggregated by state, with a simple bar chart and a cross-reference to nearby oil & gas wells for context.

In [ ]:
%pip install --quiet 'sondio[geo]>=0.1.2,<0.2' matplotlib

import sondio
# sondio >= 0.1.2 reads your key from Colab Secrets (🔑 sidebar),
# Kaggle Secrets, SONDIO_API_KEY env var, or ~/.sondio/config — in that
# order. Set whichever fits your environment.
print(f"sondio {sondio.__version__}")

In [ ]:
# Past several years of reported incidents.
incidents = sondio.us.phmsa.pipeline_incidents(all_pages=True)
print(f"{len(incidents)} incidents")
incidents.head()

In [ ]:
summary = (
    incidents.groupby("state", dropna=True)
             .agg(count=("id", "size"),
                  injuries=("injuries", "sum"),
                  fatalities=("fatalities", "sum"),
                  property_damage=("property_damage", "sum"))
             .sort_values("count", ascending=False)
             .head(15)
)
summary

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 5))
summary["count"].plot(kind="bar", ax=ax, color="#6a8caf")
ax.set_ylabel("incidents")
ax.set_xlabel("state")
ax.set_title("PHMSA pipeline incidents — top 15 states")
plt.tight_layout(); plt.show()

In [ ]:
# Drill into one state: Texas.
tx = incidents.loc[incidents["state"] == "TX"].copy()
tx = tx.dropna(subset=["latitude", "longitude"])
print(f"{len(tx)} TX incidents with coordinates")
tx[["incident_date", "operator_name", "city", "cause", "property_damage"]].head(10)

## Next steps
- Filter by `cause` to isolate corrosion vs excavation damage.
- Join with `sondio.oilgas.wells(state="TX")` to cluster incidents near producing assets.
- Compare against `sondio.earthquakes(state="TX")` to surface seismic-adjacent ruptures.